In [1]:
import json
import urllib.request
import time
from pathlib import Path
from IPython.display import display, Markdown

# =========================================================
# CONFIG
# =========================================================
BASE_DIR     = Path(".")
TEXTS_DIR    = BASE_DIR / "data_texts"
FR_PATH      = TEXTS_DIR / "20000lieues_fr.txt"
JSON_PATH    = BASE_DIR / "Points-escales-chapitres.json"
OLLAMA_URL   = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "mistral"

TEST_ESCALES = ["Île de Crespo", "Archipel des Pomotou", "Méditerranée"]

fr_text_full = FR_PATH.read_text(encoding="utf-8", errors="ignore")
fr_text_full = fr_text_full.replace("\r\n", "\n").replace("\r", "\n")

with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"✅ {len(fr_text_full):,} caractères chargés")
print(f"✅ {len(data)} escales dans le JSON")

# =========================================================
# EXTRACTION DU CHAPITRE
# =========================================================
ROMAINS_FR = {
    1:"PREMIER",2:"II",3:"III",4:"IV",5:"V",6:"VI",7:"VII",
    8:"VIII",9:"IX",10:"X",11:"XI",12:"XII",13:"XIII",14:"XIV",
    15:"XV",16:"XVI",17:"XVII",18:"XVIII",19:"XIX",20:"XX",
    21:"XXI",22:"XXII",23:"XXIII",24:"XXIV"
}

def extraire_chapitre(texte, partie, numero):
    romain = ROMAINS_FR.get(numero, str(numero))
    marqueur = f"CHAPITRE {romain}"
    split = texte.split("FIN DE LA PREMIÈRE PARTIE")
    zone = split[0] if partie == 1 else (split[1] if len(split) > 1 else texte)
    idx = zone.find(marqueur)
    if idx == -1:
        return None
    idx_fin = zone.find("CHAPITRE", idx + len(marqueur))
    return zone[idx:idx_fin].strip() if idx_fin != -1 else zone[idx:].strip()

# =========================================================
# OLLAMA
# =========================================================
def appeler_ollama(prompt):
    payload = json.dumps({
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.1, "num_predict": 1500}
    }).encode("utf-8")
    req = urllib.request.Request(
        OLLAMA_URL, data=payload,
        headers={"Content-Type": "application/json"}, method="POST"
    )
    try:
        with urllib.request.urlopen(req, timeout=180) as resp:
            return json.loads(resp.read().decode("utf-8")).get("response", "").strip()
    except Exception as e:
        return f"Erreur : {e}"

def prompt_especes(nom_escale, texte):
    return f"""Tu es un biologiste marin expert de Jules Verne.

Voici un extrait de "Vingt mille lieues sous les mers" — escale : {nom_escale}

TEXTE :
{texte[:10000]}

TÂCHE : Liste toutes les espèces marines (animaux et plantes vivant dans la mer) citées dans ce texte.

RÈGLES :
1. Copie EXACTEMENT le nom tel qu'il est écrit par Jules Verne
2. Ajoute le nom latin scientifique entre parenthèses si tu le connais
3. La citation doit être le passage EXACT du texte contenant le nom de l'espèce
4. Si aucune espèce marine n'est citée, écris uniquement : "Aucune espèce marine citée dans ce passage."
5. Ne jamais mettre [Nom inconnu] dans une citation
6. Réponse en français uniquement

FORMAT :
━━ FAUNE MARINE ━━
• [nom Jules Verne] ([nom latin])
  Jules Verne écrit : "[citation exacte courte contenant le nom]"

━━ FLORE MARINE ━━
• [nom Jules Verne] ([nom latin])
  Jules Verne écrit : "[citation exacte courte contenant le nom]"

N'affiche que les sections qui ont des espèces.
"""

# =========================================================
# TEST SUR 3 ESCALES
# =========================================================
for escale in data:
    nom_fr = escale.get("Escales du Nautilus", {}).get("fr", "")
    if nom_fr not in TEST_ESCALES:
        continue
    ch_info = escale.get("chapitre")
    if not ch_info:
        continue

    partie = ch_info["partie"]
    numero = ch_info["chapitre"]
    ch_fr = extraire_chapitre(fr_text_full, partie, numero)

    print(f"\n🧭 {nom_fr} — Partie {partie}, Ch.{numero} ({len(ch_fr) if ch_fr else 0} car.)")
    print("🤖 Envoi à Ollama...")

    reponse = appeler_ollama(prompt_especes(nom_fr, ch_fr)) if ch_fr else "Chapitre introuvable"

    display(Markdown(f"### 🧭 {nom_fr}\n{reponse}\n---"))
    time.sleep(2)

print("\n✅ Test terminé !")


✅ 908,396 caractères chargés
✅ 32 escales dans le JSON

🧭 Île de Crespo — Partie 1, Ch.15 (16671 car.)
🤖 Envoi à Ollama...


### 🧭 Île de Crespo
━━ FAUNE MARINE ━━
• Jambonneaux (spécimen non identifié)
  Jules Verne écrit : "Leur nature provoqua plus d'une fois les réflexions de Conseil. Je lui appris qu'ils étaient fabriqués avec les filaments lustrés et soyeux qui rattachent aux rochers les «jambonneaux,» sortes de coquilles très-abondantes sur les rivages de la Méditerranée."

• Cladostèphes verticillées (Cladostephus sp.)
  Jules Verne écrit : "Parmi ces précieuses hydrophytes, je trouvai des cladostèphes verticillées."

• Holoturies (Holothuria sp.)
  Jules Verne écrit : "Il se composait de divers poissons et de tranches d'holoturies."

• Aucune espèce marine citée dans ce passage.

━━ FLORE MARINE ━━
• Porphyria la (Porphyra sp.)
  Jules Verne écrit : "La _Porphyria la"
---


🧭 Archipel des Pomotou — Partie 1, Ch.19 (21056 car.)
🤖 Envoi à Ollama...


### 🧭 Archipel des Pomotou
━━ FAUNE MARINE ━━
• Maquereaux (Scomber scombrus)
  Jules Verne écrit : "Des maquereaux, des bonites, des albicores, et des variétés d'un serpent de mer nommé munérophis."

• Ostrea lamellosa
  Jules Verne écrit : "Ces mollusques appartenaient à l'_ostrea lamellosa_, qui est très-commune en Corse."

• Aucune espèce marine citée dans ce passage.

━━ FLORE MARINE ━━
• Aucune espèce marine citée dans ce passage.
---


🧭 Méditerranée — Partie 2, Ch.7 (22108 car.)
🤖 Envoi à Ollama...


### 🧭 Méditerranée
━━ FAUNE MARINE ━━
• Lamproie (Lampetra)
  Jules Verne écrit : "Des divers poissons qui l'habitent, j'ai vu les uns, entrevu les autres, sans parler de ceux que la vitesse du _Nautilus_ déroba à mes yeux. Qu'il me soit donc permis de les classer d'après cette classification fantaisiste."
• Oxyrhynque (Raja batis)
  Jules Verne écrit : "Des oxyrhinques, sortes de raies, larges de cinq pieds, au ventre blanc, au dos gris cendré et tacheté, se développaient comme de vastes châles emportés par les courants."
• Raie (Raja clavata)
  Jules Verne écrit : "D'autres raies passaient si vite que je ne pouvais reconnaître si elles méritaient ce nom d'aigles qui leur fut donné par les Grecs, ou ces qualifications de rat, de crapaud et de chauve-souris, dont les pêcheurs modernes les ont affublées."
• Squalemirande (Squalus albipinnis)
  Jules Verne écrit : "Pendant de longues heures, ils luttèrent de vitesse avec notre appareil. Je ne pouvais me lasser d'admirer ces animaux véritablement taillés pour la course."
• Gymnote (Gymnotus)
  Jules Verne écrit : "Ce sont des gymnotes-fierasfers blanchâtres qui passaient comme d'insaisissables vapeurs."
• Murène (Muraena helena)
  Jules Verne écrit : "Des murènes-congres, serpents de trois à quatre mètres enjolivés de vert, de bleu et de jaune."
• Gade (Merlangius merlangus)
  Jules Verne écrit : "Des gades-merlus, longs de trois pieds, dont le foie formait un morceau délicat."
• Céphale (Globicephala melas)
  Jules Verne écrit : "Je crois avoir reconnu en passant à l'ouvert de l'Adriatique, quelques dauphins du genre des globicéphales."
• Phoque (Monachus monachus)
  Jules Verne écrit : "Et aussi une douzaine de phoques au ventre blanc, au pelage noir, connus sous le nom de moines et qui ont absolument l'air de Dominicains longs de trois mètres."
• Tortue (Chelonia mydas)
  Jules Verne écrit : "Conseil croit avoir aperçu une tortue large de six pieds, ornée de trois arêtes saillantes dirigées longitudinalement."
• Cachalot (Physeter macrocephalus)
  Jules Verne écrit : "Je crois avoir reconnu en passant à l'ouvert de l'Adriatique, deux ou trois cachalots, munis d'une nageoire dorsale du genre des physétères."
• Tortue luth (Chelonia agassizii)
  Jules Verne écrit : "Je regrettai de ne pas avoir vu ce reptile, car, à la description que m'en fit Conseil, je crus reconnaître le luth qui forme une espèce assez rare."
• Cacouanne (Chelonia mydas)
  Jules Verne écrit : "Je ne remarquai, pour mon compte, que quelques cacouannes à carapace allongée."

━━ FAUNE MARINE - Sous-marins ━━
• Hippocampe (Hippocampus hippocampus)
  Jules Verne écrit : "Aucune espèce marine citée dans ce passage."
• Miralet (Mirapinna miranda)
  Jules Verne écrit : "Aucune espèce marine citée dans ce passage."
• Baliste (Balistes capriscus)
  Jules Verne écrit : "Aucune espèce marine citée dans ce passage."
• Tétrodon (Tetraodon fluviatilis)
  Jules Verne écrit : "Aucune espèce marine citée dans ce passage."
• Hippocampe (Hippocampus hippocampus)
  Jules Verne écrit : "Aucune espèce marine citée dans ce passage."
• Jouan (Zeus faber)
  Jules Verne écrit : "Aucune espèce marine citée dans ce passage."
• Centrise (Centriscus scutatus)
  Jules Verne écrit : "Aucune espèce marine citée dans ce passage."
• Blennie (Blennius pholis)
  Jules Verne écrit : "Aucune espèce marine citée dans ce passage."
• Surmulet (Sarpa salpa)
  Jules Verne écrit : "Aucune espèce marine citée dans ce passage."
• Labre (Labrus bergylta)
  Jules Verne écrit : "Aucune espèce marine citée dans ce passage."
• Éperlan (Stenotomus chrysops)
  Jules Verne écrit : "Aucune espèce marine citée dans ce passage."
• Exocet (Scomber japonicus)
  Jules Verne écrit : "Aucune espèce marine citée dans ce passage."
• Anchois (Engraulis encrasicolus)
  Jules Verne écrit : "Aucune espèce marine citée dans ce passage."
• Pagel (Pagellus acarne)
  Jules Verne écrit : "Aucune espèce marine citée dans ce passage."
• Bogue (Boops boops)
  Jules Verne écrit : "Aucune espèce marine citée dans ce passage."
• Orphé (Orphinus orphinus)
  Jules Verne écrit : "Aucune espèce marine citée dans ce passage."
• Limande (Limanda
---


✅ Test terminé !
